In [1]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

/Users/lakshminarayananallamothu/Desktop/MAN/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "bert-base-uncased"

label_list = ["O", "B-SYMPTOM", "I-SYMPTOM", "B-DURATION", "I-DURATION"]
label_map = {label: i for i, label in enumerate(label_list)}

print(label_map)

{'O': 0, 'B-SYMPTOM': 1, 'I-SYMPTOM': 2, 'B-DURATION': 3, 'I-DURATION': 4}


In [3]:
dataset = load_dataset("json", data_files={"train": "../data/annotated/train.json"})

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 381
    })
})
{'tokens': [' ', 'i', 'was', 'diagnosed', 'with', 'pneumonia', '.', 'i', 'ca', "n't", 'breathe', 'easily', '.'], 'ner_tags': ['O', 'O', 'O', 'B-SYMPTOM', 'O', 'B-SYMPTOM', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


In [4]:
#converting labels to ID's
def encode_labels(example):
    example["ner_tags"] = [label_map[label] for label in example["ner_tags"]]
    return example

dataset = dataset.map(encode_labels)

In [5]:
#loading tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    word_ids = tokenized.word_ids()
    labels = []
    previous_word = None

    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        elif word_id != previous_word:
            labels.append(example["ner_tags"][word_id])
        else:
            labels.append(example["ner_tags"][word_id])
        previous_word = word_id

    tokenized["labels"] = labels
    return tokenized

In [7]:
dataset = dataset.map(tokenize_and_align_labels)

In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list)
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7202.44it/s]
BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you exp

In [9]:
training_args = TrainingArguments(
    output_dir="models/",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,  # reduce overfitting
    logging_steps=10,
    save_strategy="no"
)

In [10]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    data_collator=data_collator
)

In [12]:
trainer.train()

/Users/lakshminarayananallamothu/Desktop/MAN/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,1.265611
20,0.874979
30,0.777873
40,0.664776
50,0.603147
60,0.568146
70,0.541977
80,0.511867
90,0.602441


TrainOutput(global_step=96, training_loss=0.6991393268108368, metrics={'train_runtime': 18.5331, 'train_samples_per_second': 41.116, 'train_steps_per_second': 5.18, 'total_flos': 10100516354370.0, 'train_loss': 0.6991393268108368, 'epoch': 2.0})

In [14]:
model.save_pretrained("../models/")
tokenizer.save_pretrained("../models/")

print("✅ Model saved successfully!")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

✅ Model saved successfully!
